# Game context: rest, travel, altitude and the opponent

**The question.** Streamers and the schedule show back-to-backs, three games in four nights,
miles travelled, time zones, altitude and the opponent's rest beside every game, and the
matchup value carries the opponent's defence and pace; once the day's referee assignments are
posted, the crew and its tendency show too, and the sportsbooks' line shows each team's implied
points. Do any of them change (i) whether a regular plays at all and (ii) how many minutes they
get and what they produce per minute? And would a matchup read off the line beat the one built
from defence and pace?

**Why it matters.** A manager setting a daily lineup or choosing between two streamers wants to
know which of these flags is worth acting on. A second-night rest day costs a whole game; a
small dip per minute costs little. The same data answer a second question the board relies on:
once a player has missed a game, how likely is the next one missed too? The board charges a
day-to-day listing a quarter of a game, questionable half and doubtful three quarters.

## Method

- **Regulars.** Every regular-season team game a player was on the team for (from the first
  listing with the team to the last, or to the season's end for the player's last team), with
  five or more games played before it and 10 or more minutes per game so far. A game the player
  is not listed for, or listed and did not play, is a missed game. Minutes tiers use the minutes
  per game before the game.
- **Outcomes.** Missed the game (in percentage points); minutes against the player's average so
  far; and fantasy points per 30 minutes against the player's own rate so far, for games of 10+
  minutes (ESPN's default points: PTS + REB + 2 AST + 4 STL + 4 BLK + 3PM - 2 TO + 2 FGM - FGA +
  FTM - FTA).
- **Context.** Rest (back-to-back, third game in four nights, the opponent on a back-to-back) is
  derived from every team's game dates, so it covers every season; it is checked against
  `fct_team_game_context` where that table has the game. Travel, time zones and altitude come
  only from `fct_team_game_context`. The opponent's defensive rating and pace are its averages
  before the game, with last season counted as 10 games (as the matchup value does), standardised
  over the league's teams.
- **Out of sample.** One linear model per outcome, with the minutes tier as a control, is fitted
  on the earlier seasons and estimated again on the latest finished season. The effect sizes
  reported are the latest season's, with 95% intervals from resampling players; the earlier
  seasons' estimate shows whether it replicates. The share of the outcome's variance the context
  explains out of sample is measured against a model with the minutes tier alone.
- **Missed games.** The hazard of missing the next team game after missing exactly k in a row,
  for players with 10 or more games and 15 or more minutes a game so far.
- **The betting line** (`fct_game_line`, the closing line of each played game) is tested in its
  own section below, on the games that have one.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from fs import columns, connect, describe, has_table, insufficient, leagues, save_findings

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
con = connect()
PRESENT = leagues(con)
LEAGUES = ['wnba', 'nba']  # every notebook answers for both, saying so when a league has too little
LABEL = {'wnba': 'WNBA', 'nba': 'NBA'}
COLOR = {'wnba': '#b23a0a', 'nba': '#1a7f4b'}
findings = {}
print(describe(con), 'Leagues with games:', ', '.join(LABEL[lg] for lg in PRESENT) or 'none')

In [ ]:
import replay as rp

FP = {'points': 1, 'rebounds_total': 1, 'assists': 2, 'steals': 4, 'blocks': 4, 'three_pointers_made': 1,
      'turnovers': -2, 'fgm': 2, 'fga': -1, 'ftm': 1, 'fta': -1}
cols = ', '.join((*rp.GAME_COLUMNS, 'comment', 'is_home'))
games = {}
for lg in LEAGUES:
    g = con.sql(f"select {cols} from fct_player_game where league = '{lg}'").df()
    if len(g):
        g = rp.prepare(g)
        g['fp'] = sum(w * g[c].fillna(0) for c, w in FP.items())
        games[lg] = g

ctx_cols = ['league', 'season', 'game_id', 'team_id', 'game_type', 'rest_days', 'is_back_to_back', 'is_3_in_4',
            'miles_travelled', 'time_zones_crossed', 'elevation_ft', 'at_altitude', 'opponent_back_to_back', 'is_home']
have = columns(con, 'fct_team_game_context')
ctx = (con.sql(f"select {', '.join(c for c in ctx_cols if c in have)} from fct_team_game_context").df()
       if set(ctx_cols) <= have else pd.DataFrame(columns=ctx_cols))
tg_cols = ['league', 'season', 'game_id', 'game_date', 'team_id', 'defensive_rating', 'pace']
ratings = (con.sql(f"select {', '.join(tg_cols)} from fct_team_game").df()
           if set(tg_cols) <= columns(con, 'fct_team_game') else pd.DataFrame(columns=tg_cols))
print({LABEL[lg]: f"{len(g):,} player-games" for lg, g in games.items()}, f"| {len(ctx):,} team-games with context",
      f"| {len(ratings):,} team-games with ratings")

## Coverage of the context fields by season

The share of each season's team-games with the field filled in, counting only games after a
team's first of the season (the first has no previous game to rest or travel from). A season
missing here has no landed schedule with arena cities; a low share names cities the venue seed
does not place.

In [ ]:
if len(ctx):
    c = ctx[ctx['game_type'] == 'regular'].copy()
    c['after_first'] = c['rest_days'].notna()
    coverage = c.groupby(['league', 'season']).apply(lambda x: pd.Series({
        'team_games': len(x),
        'rest_days': x['rest_days'].notna().mean(),
        'miles_travelled': x.loc[x['after_first'], 'miles_travelled'].notna().mean() if x['after_first'].any() else np.nan,
        'time_zones_crossed': x.loc[x['after_first'], 'time_zones_crossed'].notna().mean() if x['after_first'].any() else np.nan,
        'elevation_ft': x['elevation_ft'].notna().mean(),
    }), include_groups=False)
    display(coverage.round(3))
else:
    coverage = pd.DataFrame()
    print('This build has no fct_team_game_context rows.')

In [ ]:
def team_schedule(g):
    """Each regular-season team game with rest derived from the team's game dates."""
    s = g[g['regular']][['season', 'team_id', 'game_id', 'game_date']].drop_duplicates(['season', 'team_id', 'game_id'])
    s = s.sort_values(['season', 'team_id', 'game_date', 'game_id'])
    prev = s.groupby(['season', 'team_id'])['game_date'].shift(1)
    s['rest'] = (s['game_date'] - prev).dt.days - 1
    s['b2b'] = (s['rest'] == 0).astype(float)
    back2 = s.groupby(['season', 'team_id'])['game_date'].shift(2)
    s['three_in_four'] = ((s['game_date'] - back2).dt.days <= 3).astype(float)
    opp = s[['season', 'game_id', 'team_id', 'b2b', 'rest']].rename(columns={'team_id': 'opponent_id', 'b2b': 'opp_b2b', 'rest': 'opp_rest'})
    s = s.merge(opp, on=['season', 'game_id'])
    return s[s['team_id'] != s['opponent_id']]


def opponent_ratings(lg):
    """Each team's defence and pace before a game, last season counted as 10 games, standardised."""
    t = ratings[ratings['league'] == lg].copy()
    if t.empty:
        return pd.DataFrame(columns=['season', 'game_id', 'team_id', 'opp_def_z', 'pace_z'])
    t['game_date'] = pd.to_datetime(t['game_date'])
    t = t.sort_values(['season', 'team_id', 'game_date', 'game_id'])
    seasons = sorted(t['season'].unique())
    prev_mean = t.groupby(['season', 'team_id'])[['defensive_rating', 'pace']].mean()
    out = []
    for season, part in t.groupby('season'):
        part = part.copy()
        grp = part.groupby('team_id')
        n = grp.cumcount()
        earlier = [s for s in seasons if s < season]
        for c in ['defensive_rating', 'pace']:
            to_date = grp[c].transform(lambda x: x.shift(1).expanding().mean())
            last = part['team_id'].map(prev_mean.loc[earlier[-1], c]) if earlier else pd.Series(np.nan, index=part.index)
            blended = (n * to_date.fillna(0) + 10 * last) / (n + 10)
            part[c + '_asof'] = blended.where(last.notna(), to_date)
        out.append(part)
    t = pd.concat(out)
    for c, z in [('defensive_rating_asof', 'opp_def_z'), ('pace_asof', 'pace_z')]:
        t[z] = t.groupby('season')[c].transform(lambda x: (x - x.mean()) / x.std())
    return t[['season', 'game_id', 'team_id', 'opp_def_z', 'pace_z']]


def regulars(lg):
    """Every team game a player was on the team for, with what the player had done before it."""
    g = games[lg]
    reg = g[g['regular']]
    sched = team_schedule(g)
    rows = reg[['season', 'player_id', 'team_id', 'game_id', 'game_date', 'played', 'minutes', 'comment', 'fp']
               + (['started'] if 'started' in reg else [])]
    stints = rows.groupby(['season', 'player_id', 'team_id'])['game_date'].agg(first='min', last='max').reset_index()
    last_team = rows.sort_values(['game_date', 'game_id']).groupby(['season', 'player_id'])['team_id'].last()
    stints['last_team'] = list(zip(stints['season'], stints['player_id'], strict=True))
    stints['last_team'] = stints['last_team'].map(last_team) == stints['team_id']
    team_end = sched.groupby(['season', 'team_id'])['game_date'].max().rename('team_end').reset_index()
    stints = stints.merge(team_end, on=['season', 'team_id'])
    stints['end'] = stints['team_end'].where(stints['last_team'], stints['last'])
    m = stints[['season', 'player_id', 'team_id', 'first', 'end']].merge(sched, on=['season', 'team_id'])
    m = m[(m['game_date'] >= m['first']) & (m['game_date'] <= m['end'])]
    m = m.merge(rows.drop(columns=['game_date']), on=['season', 'player_id', 'team_id', 'game_id'], how='left')
    m['listed'] = m['played'].notna()
    m['played'] = m['played'].fillna(False).astype(bool)
    m['missed'] = (~m['played']).astype(float)
    m = m.sort_values(['season', 'player_id', 'game_date', 'game_id']).reset_index(drop=True)
    key = [m['season'], m['player_id']]
    mins = m['minutes'].where(m['played'], 0).fillna(0)
    fp = m['fp'].where(m['played'], 0).fillna(0)
    m['pre_games'] = m['played'].astype(int).groupby(key).cumsum() - m['played'].astype(int)
    m['pre_min'] = mins.groupby(key).cumsum() - mins
    m['pre_fp'] = fp.groupby(key).cumsum() - fp
    m['pre_mpg'] = m['pre_min'] / m['pre_games'].replace(0, np.nan)
    m['tier'] = pd.cut(m['pre_mpg'], [10, 20, 28, 34, 60], right=False, labels=['10-20', '20-28', '28-34', '34+'])
    m = m.merge(opponent_ratings(lg).rename(columns={'team_id': 'opponent_id'}), on=['season', 'game_id', 'opponent_id'], how='left')
    c = ctx[ctx['league'] == lg]
    if len(c):
        # visiting at altitude: the home team plays there every other night and is used to it
        c = c.assign(miles_1000=c['miles_travelled'] / 1000, tz_abs=c['time_zones_crossed'].abs(),
                     altitude=(c['at_altitude'].astype(bool) & ~c['is_home'].astype(bool)).astype(float))
        m = m.merge(c[['season', 'game_id', 'team_id', 'miles_1000', 'tz_abs', 'altitude', 'is_back_to_back']],
                    on=['season', 'game_id', 'team_id'], how='left')
    else:
        m = m.assign(miles_1000=np.nan, tz_abs=np.nan, altitude=np.nan, is_back_to_back=np.nan)
    homes = reg[['season', 'game_id', 'team_id', 'is_home']].drop_duplicates(['season', 'game_id', 'team_id'])
    m = m.merge(homes, on=['season', 'game_id', 'team_id'], how='left')
    m['home'] = m['is_home'].astype(float)
    m['alt_b2b'] = m['altitude'] * m['b2b'] if 'b2b' in m else np.nan
    return m.sort_values(['season', 'player_id', 'game_date', 'game_id']).reset_index(drop=True)


frames = {lg: regulars(lg) for lg in games}
for lg, m in frames.items():
    check = m.dropna(subset=['is_back_to_back'])
    agree = (check['b2b'] == check['is_back_to_back'].astype(float)).mean() if len(check) else np.nan
    print(f"{LABEL[lg]}: {len(m):,} player-team-games; back-to-backs derived from dates agree with "
          f"fct_team_game_context on {agree:.1%} of {len(check):,}" if len(check) else f"{LABEL[lg]}: {len(m):,} player-team-games")

In [ ]:
REST = {'b2b': 'Second night of a back-to-back', 'three_in_four': 'Third game in four nights',
        'opp_b2b': 'Opponent on a back-to-back', 'home': 'At home',
        'opp_def_z': 'Opponent defence, per SD (higher rating = leakier)', 'pace_z': 'Opponent pace, per SD'}
TRAVEL = {'miles_1000': 'Travel, per 1,000 miles', 'tz_abs': 'Time zones crossed, each',
          'altitude': 'Visiting at altitude (4,000 ft+)', 'alt_b2b': 'Visiting at altitude on a back-to-back (extra)'}
OUTCOMES = {'missed': ('Missed the game (pts)', 100), 'minutes_diff': ('Minutes vs own average', 1),
            'fp30_diff': ('Fantasy pts per 30 min vs own rate', 1)}


def model_rows(m, outcome):
    r = m[(m['pre_games'] >= 5) & m['tier'].notna()].copy()
    r['minutes_diff'] = r['minutes'] - r['pre_mpg']
    r['fp30_diff'] = 30 * (r['fp'] / r['minutes'] - r['pre_fp'] / r['pre_min'])
    if outcome != 'missed':
        r = r[r['played']]
    if outcome == 'fp30_diff':
        r = r[r['minutes'] >= 10]
    return r


def design(r, feats):
    tiers = pd.get_dummies(r['tier'], drop_first=True, dtype=float)
    return np.c_[np.ones(len(r)), r[feats].to_numpy(dtype=float), tiers.to_numpy()], np.ones(len(r))


def ols(X, y, w=None):
    if w is not None:
        s = np.sqrt(w)
        X, y = X * s[:, None], y * s
    return np.linalg.lstsq(X, y, rcond=None)[0]


def effects(m, feats, outcome, test_season, reps=100, seed=11):
    r = model_rows(m, outcome).dropna(subset=feats + [outcome])
    train, test = r[r['season'] < test_season], r[r['season'] == test_season]
    out = {'n_train': len(train), 'n_test': len(test)}
    if len(test) < 200:
        return out
    feats = [f for f in feats if test[f].astype(float).std() > 0]  # e.g. no WNBA venue is at altitude
    scale = OUTCOMES[outcome][1]
    Xt, _ = design(test, feats)
    yt = test[outcome].to_numpy(dtype=float)
    beta_test = ols(Xt, yt)
    rng = np.random.default_rng(seed)
    ids = test['player_id'].to_numpy()
    uniq, inv = np.unique(ids, return_inverse=True)
    draws = np.array([ols(Xt, yt, rng.poisson(1.0, len(uniq))[inv].astype(float)) for _ in range(reps)])
    for i, f in enumerate(feats, start=1):
        out[f] = (beta_test[i] * scale, *(np.quantile(draws[:, i], [0.025, 0.975]) * scale))
    if len(train) >= 200:
        Xr, _ = design(train, feats)
        yr = train[outcome].to_numpy(dtype=float)
        beta_train = ols(Xr, yr)
        for i, f in enumerate(feats, start=1):
            out[f + '_train'] = beta_train[i] * scale
        base = ols(np.delete(Xr, range(1, len(feats) + 1), axis=1), yr)
        pred_full = Xt @ beta_train
        pred_base = np.delete(Xt, range(1, len(feats) + 1), axis=1) @ base
        out['oos_r2_gain'] = 1 - ((yt - pred_full) ** 2).sum() / ((yt - pred_base) ** 2).sum()
    return out


def finished(g):
    seasons = sorted(g['season'].unique())
    playoffs = set(g.loc[~g['regular'], 'season'])
    return [s for i, s in enumerate(seasons) if s in playoffs or i < len(seasons) - 1]


results, tests = {}, {}
for lg, m in frames.items():
    done = [s for s in finished(games[lg]) if (m['season'] == s).any()]
    if not done:
        continue
    test = done[-1]
    tests[lg] = test
    for outcome in OUTCOMES:
        rest = effects(m, list(REST), outcome, test)
        travel_ok = m.loc[m['season'] == test, 'miles_1000'].notna().mean() > 0.5
        travel = effects(m, list(REST) + list(TRAVEL), outcome, test) if travel_ok else {}
        results[(lg, outcome)] = {'rest': rest, 'travel': travel}

rows = []
for (lg, outcome), res in results.items():
    for group, feats in [('rest', REST), ('travel', TRAVEL)]:
        e = res[group]
        for f, label in feats.items():
            if f in e:
                est, lo, hi = e[f]
                rows.append({'league': lg, 'outcome': OUTCOMES[outcome][0], 'context': label, 'estimate': est,
                             'lo': lo, 'hi': hi, 'earlier_seasons': e.get(f + '_train'), 'n_test': e['n_test']})
effects_table = pd.DataFrame(rows)
if len(effects_table):
    for lg in tests:
        print(f"{LABEL[lg]}: effects in {tests[lg]}, earlier seasons alongside")
        display(effects_table[effects_table['league'] == lg].drop(columns='league').round(3))
    gains = pd.DataFrame([{'league': lg, 'outcome': OUTCOMES[o][0], 'model': grp, 'oos_r2_gain_pct': 100 * r[grp].get('oos_r2_gain', np.nan)}
                          for (lg, o), r in results.items() for grp in ['rest', 'travel'] if r[grp]])
    print('Share of the outcome explained out of sample beyond the minutes tier (%):')
    display(gains.round(3))

In [ ]:
dnp = {}
for lg, m in frames.items():
    r = model_rows(m, 'missed')
    if len(r) < 200:
        continue
    t = r.groupby(['tier', 'b2b'], observed=True)['missed'].agg(['mean', 'size']).unstack('b2b')
    dnp[lg] = t
    print(f"{LABEL[lg]}: share of games missed by minutes tier, second night of a back-to-back (1.0) or not (0.0)")
    display(t.round(3))
if effects_table.shape[0]:
    fig, axes = plt.subplots(1, len(tests), figsize=(6.5 * len(tests), 4.5), squeeze=False)
    for ax, lg in zip(axes[0], tests, strict=False):
        e = effects_table[(effects_table['league'] == lg) & (effects_table['outcome'] == OUTCOMES['fp30_diff'][0])]
        ax.errorbar(e['estimate'], range(len(e)), xerr=[e['estimate'] - e['lo'], e['hi'] - e['estimate']], fmt='o', color=COLOR[lg])
        ax.set_yticks(range(len(e)), e['context'])
        ax.axvline(0, color='grey', lw=1)
        ax.set_xlabel('Fantasy points per 30 minutes (95% interval)')
        ax.set_title(f'{LABEL[lg]} {tests[lg]}: production per minute')
    plt.tight_layout()
    plt.show()

## Once a player has missed a game

The hazard is the share of players who miss the next team game, given they have missed exactly
k in a row just before it. The expected further games missed after a first miss adds up the
chance of each further game in the streak. A player who sat because of the coach (a DNP with the
coach's decision) is split from one sat with an injury or illness and from one not listed at all.

In [ ]:
def streaks(m):
    r = m.copy()
    key = [r['season'], r['player_id']]
    missed = r['missed'].astype(int)
    block = (missed == 0).astype(int).groupby(key).cumsum()
    run = missed.groupby([r['season'], r['player_id'], block]).cumsum()
    r['k'] = run.groupby(key).shift(1).fillna(0).astype(int)
    reason = np.select([~r['listed'], r['comment'].fillna('').str.contains('Coach', case=False),
                        r['comment'].fillna('').str.contains('Injur|Illness|Concussion|Reconditioning', case=False)],
                       ['not listed', "coach's decision", 'injury or illness'], 'other')
    r['reason'] = np.where(r['missed'] == 1, reason, 'played')
    r['last_reason'] = r.groupby(key)['reason'].shift(1)
    return r[(r['pre_games'] >= 10) & (r['pre_mpg'] >= 15)]


hazards, expected = {}, {}
for lg, m in frames.items():
    s = streaks(m)
    if len(s) < 500:
        continue
    s['k_cap'] = s['k'].clip(upper=10)
    h = s.groupby('k_cap')['missed'].agg(['mean', 'size'])
    hazards[lg] = h
    probs = [h['mean'].get(k, h['mean'].iloc[-1]) for k in range(1, 41)]
    expected[lg] = float(np.sum(np.cumprod(probs)))
    by_reason = s[s['k'] == 1].groupby('last_reason')['missed'].agg(['mean', 'size'])
    print(f"{LABEL[lg]}: expected further games missed after a first miss {expected[lg]:.1f}")
    display(pd.concat({'hazard': h['mean'], 'team-games': h['size']}, axis=1).round(3).T)
    display(by_reason.round(3))
if hazards:
    fig, ax = plt.subplots(figsize=(7, 4))
    for lg, h in hazards.items():
        ax.plot(h.index, h['mean'], marker='o', color=COLOR[lg], label=LABEL[lg])
    ax.set_xlabel('Games missed in a row before (10 = 10 or more)')
    ax.set_ylabel('Chance of missing the next game')
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()

## What an injury listing meant

Every official injury listing before a regular-season game (the last report by 5 pm Eastern on the
game's date, back to 2023-24 in the NBA and 2026 in the WNBA) with what followed: whether the
player missed that game, and how many of the team's games went by before the player played again
(fct_official_listing). The board charges a listed player the average of those games for the
status, capped at the games the team has left; before this calibration it charged a fixed share
of a game (out: every game left without a return date; doubtful 0.75, questionable 0.5,
day-to-day 0.25).

In [ ]:
OLD = {'Out': 'every game left', 'Doubtful': 0.75, 'Questionable': 0.5, 'Probable': 0.0}
calibration = {}
need = {'league', 'status', 'reason_group', 'played', 'games_missed'}
if need <= columns(con, 'fct_official_listing'):
    ol = con.sql("select league, status, played, games_missed, team_games_left from fct_official_listing "
                 "where reason_group = 'injury'").df()
    for lg in LEAGUES:
        o = ol[ol['league'] == lg]
        if o.empty:
            continue
        t = o.groupby('status').agg(listings=('played', 'size'), missed_game=('played', lambda x: 1 - x.mean()),
                                    games_missed_mean=('games_missed', 'mean'), games_missed_median=('games_missed', 'median'))
        t = t.reindex([x for x in ['Out', 'Doubtful', 'Questionable', 'Probable', 'Available'] if x in t.index])
        t['old_charge'] = t.index.map(OLD)
        calibration[lg] = t
        print(LABEL[lg], f'({len(o):,} injury listings)')
        display(t.round(2))
if not calibration:
    print('No official injury listing in this build lines up with a played game yet.')

ASSUMED = {'out': 1.0, 'doubtful': 0.75, 'questionable': 0.5, 'day_to_day': 0.25}
listing = {}
need = {'league', 'report_date', 'stats_player_id', 'status_group'}
if need <= columns(con, 'fct_availability'):
    av = con.sql("select league, report_date, stats_player_id as player_id, status_group from fct_availability "
                 "where stats_player_id is not null").df()
    av['report_date'] = pd.to_datetime(av['report_date'])
    for lg, m in frames.items():
        a = av[av['league'] == lg]
        if a.empty:
            continue
        nxt = m[['player_id', 'game_date', 'missed']].sort_values('game_date')
        a = a.sort_values('report_date')
        j = pd.merge_asof(a, nxt, left_on='report_date', right_on='game_date', by='player_id', direction='forward')
        j = j[(j['game_date'] - j['report_date']).dt.days <= 3].drop_duplicates(['player_id', 'game_date'], keep='last')
        if len(j):
            t = j.groupby('status_group')['missed'].agg(['mean', 'size'])
            t['board_assumes'] = t.index.map(ASSUMED)
            listing[lg] = t
            print(LABEL[lg])
            display(t.round(3))
if not listing:
    print('No injury listing in this build lines up with a game yet.')

## The referee crew

Every game's crew comes from ESPN's game summaries (`fct_game_official`); the day's crews are
posted by the leagues at about 9 am Eastern and shown beside the schedule. Does a crew that has
called more fouls and free throws, or run faster games, change what a regular does?

- **Crew tendency, point in time.** For each official, the average game they worked before this
  one (all earlier games, this season and before), as the difference from the league-season's
  average game in total free-throw attempts, fouls and pace (both teams, `fct_team_game`),
  shrunk toward zero as if the official had also worked 20 average games (the same shrinkage
  `mart_schedule_upcoming` applies). The crew's tendency is the average of its officials'. An
  alternate who did not work the game is not counted.
- **First, the game.** Does the crew's tendency before a game predict that game's total
  free-throw attempts, fouls and pace in the latest finished season? The slope is the change in
  the game's figure per unit of tendency (1.0 would mean the tendency carries over in full).
- **Then the player,** in the same effects framework as above: personal fouls, free-throw
  attempts and fantasy points per 30 minutes against the player's own rate so far (games of 10+
  minutes), on the crew's free-throw and pace tendencies, with the minutes tier as a control;
  fitted on the earlier seasons and estimated again on the latest finished season, with 95%
  intervals from resampling players and the share of the outcome explained out of sample.

In [ ]:
REF_PRIOR = 20  # games of an average official every official's record starts from
CREW = {'crew_fta': 'Crew free-throw tendency, per attempt a game', 'crew_pace': 'Crew pace tendency, per possession'}
CREW_OUTCOMES = {'pf30_diff': 'Personal fouls per 30 min vs own rate', 'fta30_diff': 'Free-throw attempts per 30 min vs own rate',
                 'fp30_diff': OUTCOMES['fp30_diff'][0]}
OUTCOMES.update({k: (v, 1) for k, v in CREW_OUTCOMES.items() if k not in OUTCOMES})
GAME_FIGURES = {'fta': 'free-throw attempts', 'fouls': 'personal fouls', 'pace': 'pace'}


def crew_games(lg):
    """Each regular-season game's figures against the league-season average, and its crew's tendency before it."""
    cols = ['season', 'game_id', 'fta', 'fouls', 'pace', 'fta_delta', 'fouls_delta', 'pace_delta', 'crew_fta', 'crew_fouls', 'crew_pace', 'crew_prior_games']
    if not ({'official_key', 'game_id', 'role'} <= columns(con, 'fct_game_official')
            and {'fouls_personal', 'free_throws_attempted', 'pace'} <= columns(con, 'fct_team_game')):
        return pd.DataFrame(columns=cols)
    t = con.sql(f"""select season, game_id, min(game_date) as game_date, sum(fouls_personal) as fouls,
                         sum(free_throws_attempted) as fta, avg(pace) as pace
                  from fct_team_game where league = '{lg}' and substr(game_id, 3, 1) = '2'
                  group by season, game_id having count(*) = 2""").df()
    o = con.sql(f"select game_id, official_key, role from fct_game_official where league = '{lg}'").df()
    o = o[o['role'].fillna('') != 'alternate']
    if t.empty or o.empty:
        return pd.DataFrame(columns=cols)
    t['game_date'] = pd.to_datetime(t['game_date'])
    for c in GAME_FIGURES:
        t[c + '_delta'] = t[c] - t.groupby('season')[c].transform('mean')
    j = o.merge(t, on='game_id').sort_values(['official_key', 'game_date', 'game_id'])
    grp = j.groupby('official_key')
    j['prior_n'] = grp.cumcount()
    for c in GAME_FIGURES:
        # the official's earlier games only: sum / (n + prior) is their average shrunk toward zero
        j['tend_' + c] = (grp[c + '_delta'].cumsum() - j[c + '_delta']) / (j['prior_n'] + REF_PRIOR)
    crew = j.groupby('game_id').agg(crew_fta=('tend_fta', 'mean'), crew_fouls=('tend_fouls', 'mean'),
                                    crew_pace=('tend_pace', 'mean'), crew_prior_games=('prior_n', 'sum'))
    return t.merge(crew, on='game_id')[cols]


crew_by_game = {lg: crew_games(lg) for lg in games}
game_rows, crew_rows, crew_results = [], [], {}
for lg, cg in crew_by_game.items():
    if lg not in tests or cg.empty:
        print(f"{LABEL[lg]}: no referee crews in this build yet" if cg.empty else f"{LABEL[lg]}: no finished season to test on")
        continue
    test = tests[lg]
    g = cg[(cg['season'] == test) & (cg['crew_prior_games'] >= 30)]
    for c, label in GAME_FIGURES.items():
        x, y = g['crew_' + c].astype(float), g[c + '_delta'].astype(float)
        if len(g) >= 50 and x.std() > 0:
            slope = float(np.polyfit(x, y, 1)[0])
            game_rows.append({'league': lg, 'season': test, 'game figure': label, 'games': len(g),
                              'tendency SD': x.std(), 'game SD': y.std(), 'slope': slope, 'correlation': x.corr(y)})
    m = frames[lg].merge(cg[['season', 'game_id', 'crew_fta', 'crew_pace', 'crew_prior_games']], on=['season', 'game_id'], how='left')
    box = (con.sql(f"select season, game_id, player_id, fouls_personal, free_throws_attempted from fct_player_game where league = '{lg}'").df()
           if {'fouls_personal', 'free_throws_attempted'} <= columns(con, 'fct_player_game') else pd.DataFrame())
    if box.empty:
        continue
    m = m.merge(box, on=['season', 'game_id', 'player_id'], how='left')
    key = [m['season'], m['player_id']]
    mins = m['minutes'].where(m['played'], 0).fillna(0)
    for src, out in [('fouls_personal', 'pf30_diff'), ('free_throws_attempted', 'fta30_diff')]:
        v = pd.to_numeric(m[src], errors='coerce').where(m['played'], 0).fillna(0)
        pre = v.groupby(key).cumsum() - v
        m[out] = (30 * (v / m['minutes'] - pre / m['pre_min'])).where(m['played'] & (mins >= 10))
    m.loc[m['crew_prior_games'] < 30, list(CREW)] = np.nan  # a crew with too little history says nothing yet
    crew_results[lg] = {o: effects(m, list(CREW), o, test) for o in CREW_OUTCOMES}
    for o, e in crew_results[lg].items():
        for f, label in CREW.items():
            if f in e:
                est, lo, hi = e[f]
                crew_rows.append({'league': lg, 'outcome': CREW_OUTCOMES[o], 'context': label, 'estimate': est, 'lo': lo, 'hi': hi,
                                  'earlier_seasons': e.get(f + '_train'), 'oos_r2_gain_pct': 100 * e.get('oos_r2_gain', np.nan),
                                  'n_test': e['n_test']})
crew_game_table, crew_table = pd.DataFrame(game_rows), pd.DataFrame(crew_rows)
if len(crew_game_table):
    print("Does the crew's tendency before a game predict the game? (latest finished season, crews with 30+ prior games)")
    display(crew_game_table.round(3))
if len(crew_table):
    print("Crew tendency and the player's game, per unit of tendency (latest finished season, earlier seasons alongside):")
    display(crew_table.round(3))
elif not len(crew_game_table):
    print('This build has no referee crews lined up with played games yet.')

## The betting line

`fct_game_line` keeps the closing line of every played game whose ESPN summary carried one: the
home spread, the over/under, and the points each team is implied to score, (total ∓ spread) / 2.
The schedule shows the line beside the matchup value; the matchup does not use it. Three questions
come before it could:

- **Coverage.** The share of each league-season's played games with a line. A season whose
  summaries carry no line shows a low share, and the tests use only games with a line.
- **Does the line add to defence and pace?** The team's implied points, standardised over the
  league-season's team-games with a line, and the absolute spread per 5 points (blowout risk: a
  lopsided game sends starters to the bench for the fourth quarter) join the opponent's defence and
  pace in the effects framework above, for fantasy points per 30 minutes and minutes against the
  player's own figures, with the minutes tier as a control. Regulars are also split by whether they
  started. Each effect is the test part's, with 95% intervals from resampling players and the fit
  part's estimate alongside; the gain is the share of the outcome explained out of sample beyond
  defence, pace and the minutes tier, with its interval.
- **Which matchup predicts better?** The current matchup (`fantasy_matchup`: the opponent's
  defence and the average of both teams' pace, as `mart_schedule_upcoming` computes it) against a
  line matchup (the team's implied points less the league-season average), each given one slope on
  the fit part, at predicting a regular's fantasy points in the game against their own average so
  far, and per 30 minutes. Both are scored out of sample on the same test games, as the share
  explained beyond the minutes tier.
- **Fit and test.** Earlier seasons with lines fit and the latest finished season tests, as
  above. When the latest finished season is the only one with enough lines, its games split at
  the median date: the first half fits, the second tests. With fewer than 500 player-games with a
  line on either side, the league is reported as accumulating.

In [ ]:
LINE_COLS = ['league', 'season', 'game_id', 'home_team_id', 'away_team_id', 'home_spread', 'over_under',
             'home_implied_points', 'away_implied_points']
MIN_LINE_ROWS = 500  # player-games with a line on each side of the split before the line is tested
gl = (con.sql(f"select {', '.join(LINE_COLS)} from fct_game_line where home_spread is not null and over_under is not null").df()
      if set(LINE_COLS) <= columns(con, 'fct_game_line') else pd.DataFrame(columns=LINE_COLS))


def as_key(s):
    """Ids as text for joining, whether they arrive as numbers or as strings."""
    n = pd.to_numeric(s, errors='coerce')
    return n.astype('Int64').astype('string') if n.notna().sum() == s.notna().sum() else s.astype('string')


cov = []
for lg, g in games.items():
    lined = set(gl.loc[gl['league'] == lg, 'game_id'].astype(str))
    p = g[g['played']].drop_duplicates('game_id')[['season', 'game_id', 'game_date', 'regular']].copy()
    p['line'] = p['game_id'].astype(str).isin(lined)
    for season, x in p.groupby('season'):
        reg = x[x['regular']]
        cov.append({'league': lg, 'season': season, 'played_games': len(x), 'with_line': int(x['line'].sum()),
                    'share': x['line'].mean(), 'regular_share': reg['line'].mean() if len(reg) else np.nan,
                    'first_line': x.loc[x['line'], 'game_date'].min()})
line_coverage = pd.DataFrame(cov)
if gl.empty:
    print('This build has no closing lines (fct_game_line) yet.')
if len(line_coverage):
    print('Share of played games with a closing line, per league-season:')
    display(line_coverage.round({'share': 3, 'regular_share': 3}))

In [ ]:
import yaml

MATCHUP = yaml.safe_load(rp.PROJECT_FILE.read_text(encoding='utf-8'))['vars']['fantasy_matchup']
LINE = {'implied_z': 'Team implied points, per SD', 'blowout_5': 'Absolute spread, per 5 points'}
BASE = {'opp_def_z': REST['opp_def_z'], 'pace_z': REST['pace_z']}
ROLES = {'all': 'All regulars', 'starter': 'Started', 'bench': 'Came off the bench'}
LINE_OUTCOMES = {'fp30_diff': OUTCOMES['fp30_diff'][0], 'minutes_diff': OUTCOMES['minutes_diff'][0]}
MATCHUP_OUTCOMES = {'fp_diff': 'Fantasy pts in the game vs own average', 'fp30_diff': OUTCOMES['fp30_diff'][0]}
SIGNALS = {'matchup': 'Defence + pace (current matchup)', 'implied_vs_avg': 'Implied team points vs league average',
           'both': 'Both together'}


def team_lines(lg):
    """Each team's side of each line: implied points (raw, against the league-season average, standardised) and spread."""
    l = gl[gl['league'] == lg]
    cols = ['season', 'game_id', 'team_id', 'side_home', 'implied', 'spread', 'implied_vs_avg', 'implied_z']
    if l.empty:
        return pd.DataFrame(columns=cols)

    def side(home, team, implied, spread):
        return pd.DataFrame({'season': l['season'], 'game_id': l['game_id'].astype(str), 'team_id': as_key(l[team]),
                             'side_home': home, 'implied': l[implied].astype(float), 'spread': spread.astype(float)})

    t = pd.concat([side(1.0, 'home_team_id', 'home_implied_points', l['home_spread']),
                   side(0.0, 'away_team_id', 'away_implied_points', -l['home_spread'])], ignore_index=True)
    grp = t.groupby('season')['implied']
    t['implied_vs_avg'] = t['implied'] - grp.transform('mean')
    t['implied_z'] = t['implied_vs_avg'] / grp.transform('std')
    return t[cols]


def with_lines(lg, m):
    """The regulars frame with the team's side of the line, its own pace and the current matchup value."""
    t = team_lines(lg)
    feats = ['implied', 'spread', 'implied_vs_avg', 'implied_z']
    m = m.assign(_gid=m['game_id'].astype(str), _tid=as_key(m['team_id']))
    by_team = t.dropna(subset=['team_id']).drop_duplicates(['game_id', 'team_id'])
    m = m.merge(by_team.rename(columns={'game_id': '_gid', 'team_id': '_tid'})[['_gid', '_tid', *feats]],
                on=['_gid', '_tid'], how='left')
    by_side = t[t['team_id'].isna()].drop_duplicates(['game_id', 'side_home'])
    if len(by_side):  # a line whose teams did not link: take the side by home and away
        x = m[['_gid', 'home']].merge(by_side.rename(columns={'game_id': '_gid', 'side_home': 'home'})[['_gid', 'home', *feats]],
                                      on=['_gid', 'home'], how='left')
        for f in feats:
            m[f] = m[f].fillna(pd.Series(x[f].to_numpy(), index=m.index))
    m['blowout_5'] = m['spread'].abs() / 5
    own = opponent_ratings(lg)[['season', 'game_id', 'team_id', 'pace_z']].rename(columns={'pace_z': 'own_pace_z'})
    m = m.merge(own, on=['season', 'game_id', 'team_id'], how='left')
    b = MATCHUP.get(lg, {'defense': 0.0, 'pace': 0.0})
    m['matchup'] = b['defense'] * m['opp_def_z'] + b['pace'] * (m['own_pace_z'] + m['pace_z']) / 2
    return m.drop(columns=['_gid', '_tid'])


def line_model_rows(m, outcome):
    r = model_rows(m, 'minutes_diff' if outcome == 'fp_diff' else outcome)
    r['fp_diff'] = r['fp'] - r['pre_fp'] / r['pre_games']
    return r.dropna(subset=[*BASE, *LINE, 'matchup', 'implied_vs_avg', outcome])


def bootstrap_weights(test, reps=100, seed=11):
    rng = np.random.default_rng(seed)
    uniq, inv = np.unique(test['player_id'].to_numpy(), return_inverse=True)
    return [rng.poisson(1.0, len(uniq))[inv].astype(float) for _ in range(reps)]


def gain(err, err_base, w=None):
    w = np.ones(len(err)) if w is None else w
    return 1 - (w * err).sum() / (w * err_base).sum()


def line_effects(train, test, outcome):
    """Defence, pace and the line on the test part with intervals; out-of-sample gain of the line over defence + pace."""
    out = {'n_train': len(train), 'n_test': len(test)}
    if min(len(train), len(test)) < MIN_LINE_ROWS:
        return out
    feats = [*BASE, *LINE]
    Xt, _ = design(test, feats)
    Xr, _ = design(train, feats)
    yt, yr = test[outcome].to_numpy(dtype=float), train[outcome].to_numpy(dtype=float)
    w = bootstrap_weights(test)
    beta_test, beta_train = ols(Xt, yt), ols(Xr, yr)
    draws = np.array([ols(Xt, yt, wi) for wi in w])
    for i, f in enumerate(feats, start=1):
        out[f] = (beta_test[i], *np.quantile(draws[:, i], [0.025, 0.975]))
        out[f + '_train'] = beta_train[i]
    keep = np.delete(np.arange(Xr.shape[1]), range(1 + len(BASE), 1 + len(feats)))  # drop the line's columns
    err_full = (yt - Xt @ beta_train) ** 2
    err_base = (yt - Xt[:, keep] @ ols(Xr[:, keep], yr)) ** 2
    out['oos_r2_gain'] = (gain(err_full, err_base), *np.quantile([gain(err_full, err_base, wi) for wi in w], [0.025, 0.975]))
    return out


def compare_matchups(train, test, outcome):
    """Out-of-sample share explained beyond the minutes tier by each matchup signal, one slope each, on the same games."""
    out = {'n_train': len(train), 'n_test': len(test)}
    if min(len(train), len(test)) < MIN_LINE_ROWS:
        return out
    yt, yr = test[outcome].to_numpy(dtype=float), train[outcome].to_numpy(dtype=float)
    X0r, _ = design(train, [])
    X0t, _ = design(test, [])
    err0 = (yt - X0t @ ols(X0r, yr)) ** 2
    errs = {}
    for name, feats in [('matchup', ['matchup']), ('implied_vs_avg', ['implied_vs_avg']), ('both', ['matchup', 'implied_vs_avg'])]:
        Xr, _ = design(train, feats)
        Xt, _ = design(test, feats)
        errs[name] = (yt - Xt @ ols(Xr, yr)) ** 2
    w = bootstrap_weights(test, seed=7)
    for name, err in errs.items():
        out[name] = (gain(err, err0), *np.quantile([gain(err, err0, wi) for wi in w], [0.025, 0.975]))
    diff = [gain(errs['implied_vs_avg'], err0, wi) - gain(errs['matchup'], err0, wi) for wi in w]
    out['line_minus_current'] = (out['implied_vs_avg'][0] - out['matchup'][0], *np.quantile(diff, [0.025, 0.975]))
    out['corr'] = {name: float(np.corrcoef(test[name].astype(float), yt)[0, 1]) for name in ['matchup', 'implied_vs_avg']}
    return out


line_frames, line_split, line_rows, line_gains, matchup_rows = {}, {}, [], [], []
for lg, m0 in frames.items():
    if lg not in tests or gl[gl['league'] == lg].empty:
        continue
    m = with_lines(lg, m0)
    line_frames[lg] = m
    test_season = tests[lg]
    p = line_model_rows(m, 'minutes_diff')
    p = p[p['season'] <= test_season]
    early, latest = p[p['season'] < test_season], p[p['season'] == test_season]
    if len(early) >= MIN_LINE_ROWS and len(latest) >= MIN_LINE_ROWS:
        cut, note = None, f"fit on {', '.join(sorted(early['season'].unique()))}, tested on {test_season}"
    else:
        cut = latest['game_date'].median() if len(latest) else pd.NaT
        note = (f"{test_season} only: fit on games before {cut:%Y-%m-%d}, tested on games from then on"
                if pd.notna(cut) else f"no game with a line in {test_season}")
    line_split[lg] = note

    def split(r, cut=cut, test_season=test_season):
        r = r[r['season'] <= test_season]
        if cut is None:
            return r[r['season'] < test_season], r[r['season'] == test_season]
        r = r[r['season'] == test_season]
        return r[r['game_date'] < cut], r[r['game_date'] >= cut]

    for outcome, olabel in LINE_OUTCOMES.items():
        r = line_model_rows(m, outcome)
        for role, rlabel in ROLES.items():
            if role != 'all' and 'started' not in r:
                continue
            x = r if role == 'all' else r[r['started'].fillna(False).astype(bool) == (role == 'starter')]
            e = line_effects(*split(x), outcome)
            for f, flabel in {**BASE, **LINE}.items():
                if f in e:
                    est, lo, hi = e[f]
                    line_rows.append({'league': lg, 'outcome': olabel, 'regulars': rlabel, 'context': flabel, 'estimate': est,
                                      'lo': lo, 'hi': hi, 'fit_part': e[f + '_train'], 'n_test': e['n_test']})
            line_gains.append({'league': lg, 'outcome': olabel, 'regulars': rlabel, 'n_fit': e['n_train'], 'n_test': e['n_test'],
                               **({'gain_pct': 100 * e['oos_r2_gain'][0], 'lo': 100 * e['oos_r2_gain'][1], 'hi': 100 * e['oos_r2_gain'][2]}
                                  if 'oos_r2_gain' in e else {})})
    for outcome, olabel in MATCHUP_OUTCOMES.items():
        c = compare_matchups(*split(line_model_rows(m, outcome)), outcome)
        for name, slabel in SIGNALS.items():
            if name in c:
                matchup_rows.append({'league': lg, 'outcome': olabel, 'signal': slabel, 'gain_pct': 100 * c[name][0],
                                     'lo': 100 * c[name][1], 'hi': 100 * c[name][2], 'correlation': c['corr'].get(name),
                                     'n_test': c['n_test']})
        if 'line_minus_current' in c:
            d = c['line_minus_current']
            matchup_rows.append({'league': lg, 'outcome': olabel, 'signal': 'Line minus current', 'gain_pct': 100 * d[0],
                                 'lo': 100 * d[1], 'hi': 100 * d[2], 'correlation': None, 'n_test': c['n_test']})
line_table, line_gain_table, matchup_table = pd.DataFrame(line_rows), pd.DataFrame(line_gains), pd.DataFrame(matchup_rows)
for lg, note in line_split.items():
    print(f"{LABEL[lg]}: {note}")
    if len(line_table) and (line_table['league'] == lg).any():
        display(line_table[line_table['league'] == lg].drop(columns='league').round(3))
        print('Share of the outcome explained out of sample beyond defence, pace and the minutes tier (%):')
        display(line_gain_table[line_gain_table['league'] == lg].drop(columns='league').round(3))
        print('Matchup signals: share explained out of sample beyond the minutes tier (%), on the same games:')
        display(matchup_table[matchup_table['league'] == lg].drop(columns='league').round(3))
    else:
        g = line_gain_table[line_gain_table['league'] == lg] if len(line_gain_table) else line_gain_table
        n = g[['n_fit', 'n_test']].min().min() if len(g) else 0
        print(f"  not enough player-games with a line yet ({int(n):,} on the thinner side; {MIN_LINE_ROWS} needed)")
if not line_split:
    print('No league has closing lines lined up with a finished season yet.')
if len(line_table):
    shown = [lg for lg in line_split if (line_table['league'] == lg).any()]
    fig, axes = plt.subplots(1, len(shown), figsize=(6.5 * len(shown), 3.8), squeeze=False)
    for ax, lg in zip(axes[0], shown, strict=False):
        e = line_table[(line_table['league'] == lg) & (line_table['outcome'] == LINE_OUTCOMES['minutes_diff'])
                       & (line_table['context'].isin(LINE.values()))]
        labels = e['context'] + ', ' + e['regulars'].str.lower()
        ax.errorbar(e['estimate'], range(len(e)), xerr=[e['estimate'] - e['lo'], e['hi'] - e['estimate']], fmt='o', color=COLOR[lg])
        ax.set_yticks(range(len(e)), labels)
        ax.axvline(0, color='grey', lw=1)
        ax.set_xlabel('Minutes vs own average (95% interval)')
        ax.set_title(f'{LABEL[lg]}: the line and minutes')
    plt.tight_layout()
    plt.show()

In [ ]:
def eff(lg, outcome, f):
    e = effects_table[(effects_table['league'] == lg) & (effects_table['outcome'] == OUTCOMES[outcome][0])
                      & (effects_table['context'] == {**REST, **TRAVEL}[f])] if len(effects_table) else effects_table
    return e.iloc[0] if len(e) else None


def show(e, digits=2, unit=''):
    return f"{e['estimate']:+.{digits}f}{unit} ({e['lo']:+.{digits}f} to {e['hi']:+.{digits}f})"


for lg in LEAGUES:
    n = int(model_rows(frames[lg], 'missed').shape[0]) if lg in frames else 0
    if lg not in tests or eff(lg, 'fp30_diff', 'b2b') is None:
        findings[lg] = insufficient(
            f"Not enough history in this build: the {LABEL[lg]} context study needs a finished regular season "
            "of player-games, and this build has none yet.", n=n, n_label='player-team-games')
        continue
    test = tests[lg]
    b2b_fp, b2b_miss, b2b_min = eff(lg, 'fp30_diff', 'b2b'), eff(lg, 'missed', 'b2b'), eff(lg, 'minutes_diff', 'b2b')
    defence = eff(lg, 'fp30_diff', 'opp_def_z')
    travel = eff(lg, 'fp30_diff', 'miles_1000')
    h = hazards.get(lg)
    headline = (f"In {test}, the second night of a back-to-back changed a regular's chance of sitting out by "
                f"{b2b_miss['estimate']:+.1f} points and production by {b2b_fp['estimate']:+.2f} fantasy points per 30 minutes"
                + (f", while a standard deviation of opponent defence moved production by {defence['estimate']:+.2f}" if defence is not None else '')
                + (f"; after missing one game, {h['mean'].get(1, np.nan):.0%} missed the next." if h is not None else '.'))
    facts = [
        {'label': 'Back-to-back: chance of sitting out', 'value': show(b2b_miss, 1, ' pts') + (f"; earlier {b2b_miss['earlier_seasons']:+.1f}" if pd.notna(b2b_miss['earlier_seasons']) else '')},
        {'label': 'Back-to-back: minutes and fantasy pts / 30 min', 'value': f"{b2b_min['estimate']:+.2f} min; {show(b2b_fp)}"},
        {'label': 'Opponent defence, per SD: fantasy pts / 30 min', 'value': show(defence) if defence is not None else 'no team ratings in this build'},
        {'label': 'Travel, per 1,000 miles: fantasy pts / 30 min', 'value': show(travel) if travel is not None else 'no travel fields for this season'},
    ]
    if h is not None:
        facts.append({'label': 'Missed the next game after missing 1 / 3 in a row', 'value': f"{h['mean'].get(1, np.nan):.0%} / {h['mean'].get(3, np.nan):.0%}; {expected[lg]:.1f} more games expected after a first miss"})
    if lg in calibration and calibration[lg]['listings'].sum() >= 100:
        t = calibration[lg]
        facts.append({'label': 'Official listings: missed the game; games missed on average',
                      'value': '; '.join(f"{k} {v['missed_game']:.0%}, {v['games_missed_mean']:.1f} (n {int(v['listings']):,})"
                                         for k, v in t.iterrows() if k in ('Out', 'Doubtful', 'Questionable', 'Probable'))})
    elif lg in listing and listing[lg]['size'].sum() >= 30:
        t = listing[lg]
        facts.append({'label': 'Listed players who missed the next game', 'value': '; '.join(f"{k} {v['mean']:.0%} of {int(v['size'])} (board {ASSUMED.get(k, float('nan')):.2f})" for k, v in t.iterrows())})
    elif lg in listing:
        facts.append({'label': 'Injury listings matched to a next game', 'value': f"{int(listing[lg]['size'].sum())} so far: accumulating"})
    elif len(coverage) and lg in coverage.index.get_level_values(0):
        cov = coverage.loc[lg]
        facts.append({'label': 'Travel fields filled, by season', 'value': ', '.join(f"{s} {v:.0%}" for s, v in cov['miles_travelled'].items())})
    ce = crew_table[(crew_table['league'] == lg)] if len(crew_table) else crew_table
    cg = crew_game_table[(crew_game_table['league'] == lg)] if len(crew_game_table) else crew_game_table
    if len(ce):
        def crew_eff(outcome, f):
            r = ce[(ce['outcome'] == CREW_OUTCOMES[outcome]) & (ce['context'] == CREW[f])]
            return r.iloc[0] if len(r) else None
        fta_game = cg[cg['game figure'] == 'free-throw attempts']
        game_note = (f"the game's attempts moved {fta_game.iloc[0]['slope']:+.2f} per attempt of tendency; "
                     if len(fta_game) else '')
        parts = [f"{label} {show(x)}" for label, x in [('player FTA', crew_eff('fta30_diff', 'crew_fta')),
                                                         ('fouls', crew_eff('pf30_diff', 'crew_fta')),
                                                         ('fantasy pts', crew_eff('fp30_diff', 'crew_fta'))] if x is not None]
        facts.append({'label': 'Referee crew, per free throw a game of tendency (per 30 min)',
                      'value': game_note + '; '.join(parts)})
    lc = line_coverage[line_coverage['league'] == lg] if len(line_coverage) else line_coverage
    if len(lc) and lc['with_line'].sum():
        facts.append({'label': 'Played games with a closing line, by season',
                      'value': ', '.join(f"{s} {v:.0%}" for s, v in zip(lc['season'], lc['share'], strict=True))})
    lt = line_table[line_table['league'] == lg] if len(line_table) else line_table
    if len(lt):
        def line_eff(outcome, f, role='all'):
            r = lt[(lt['outcome'] == LINE_OUTCOMES[outcome]) & (lt['context'] == LINE[f]) & (lt['regulars'] == ROLES[role])]
            return r.iloc[0] if len(r) else None
        lg_gain = line_gain_table[(line_gain_table['league'] == lg) & (line_gain_table['regulars'] == ROLES['all'])].dropna(subset=['gain_pct'])
        gains_txt = '; '.join(f"{'per 30 min' if x['outcome'] == LINE_OUTCOMES['fp30_diff'] else 'minutes'} {x['gain_pct']:+.2f}% "
                              f"({x['lo']:+.2f} to {x['hi']:+.2f})" for x in lg_gain.to_dict('records'))
        parts = [f"{label} {show(x)}" for label, x in [('implied pts per SD, fantasy pts / 30 min', line_eff('fp30_diff', 'implied_z')),
                                                         ("5 pts of spread, starters' minutes", line_eff('minutes_diff', 'blowout_5', 'starter')),
                                                         ('bench minutes', line_eff('minutes_diff', 'blowout_5', 'bench'))] if x is not None]
        facts.append({'label': 'The line beyond defence and pace, out of sample',
                      'value': f"share explained {gains_txt}; " + '; '.join(parts) + f" ({line_split[lg]})"})
        mt = matchup_table[(matchup_table['league'] == lg) & (matchup_table['outcome'] == MATCHUP_OUTCOMES['fp_diff'])]
        v = {x['signal']: x for x in mt.to_dict('records')}
        if {SIGNALS['matchup'], SIGNALS['implied_vs_avg'], 'Line minus current'} <= set(v):
            cur, lin, d = v[SIGNALS['matchup']], v[SIGNALS['implied_vs_avg']], v['Line minus current']
            facts.append({'label': 'Matchup from the line against defence and pace: fantasy pts in the game explained out of sample',
                          'value': f"line {lin['gain_pct']:+.2f}%, current {cur['gain_pct']:+.2f}%; "
                                   f"difference {d['gain_pct']:+.2f} points ({d['lo']:+.2f} to {d['hi']:+.2f})"})
    elif lg in line_split:
        facts.append({'label': 'Betting line backtest', 'value': f"accumulating: too few player-games with a line yet ({line_split[lg]})"})
    e = effects_table[(effects_table['league'] == lg)]
    table = []
    for f, label in {**REST, **TRAVEL}.items():
        cells = []
        for o in ['fp30_diff', 'minutes_diff', 'missed']:
            x = eff(lg, o, f)
            cells.append(None if x is None else show(x, 1 if o == 'missed' else 2))
        if any(cells):
            table.append([label, *cells])
    for f, label in CREW.items():
        cells = []
        for o in ['fp30_diff']:
            r = ce[(ce['outcome'] == CREW_OUTCOMES[o]) & (ce['context'] == label)] if len(ce) else ce
            cells.append(show(r.iloc[0]) if len(r) else None)
        if any(cells):
            table.append([label, *cells, None, None])
    for f, label in LINE.items():
        cells = []
        for o in ['fp30_diff', 'minutes_diff']:
            r = lt[(lt['outcome'] == LINE_OUTCOMES[o]) & (lt['context'] == label) & (lt['regulars'] == ROLES['all'])] if len(lt) else lt
            cells.append(show(r.iloc[0]) if len(r) else None)
        if any(cells):
            table.append([label, *cells, None])
    findings[lg] = {
        'headline': headline,
        'facts': facts[:10],
        'table': {'columns': [f'Context ({test})', 'Fantasy pts / 30 min', 'Minutes', 'Sat out (pts)'], 'rows': table},
        'n': n,
        'n_label': 'player-team-games',
    }
save_findings('03_game_context', {'title': 'Game context', 'leagues': findings})
findings